# Text summarization mit verschiedenen Modellen für die Kapitel *Encoder and Decoder Stacks + Attention*

In der nächsten Zelle sieht man die Ausschnitte aus dem Paper, die wir behandeln sollen, ohne Bilder, (komplexe) Formeln, etc...

In [ ]:
!pip install "transformers<5.0.0"

from transformers import pipeline, set_seed

set_seed(42)

paragraph = """
Encoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two sub-layers.
The first is a multi-head self-attention mechanism, and the second is a simple, positionwise fully connected feed-forward network.
We employ a residual connection around each of the two sub-layers, followed by layer normalization.
That is, the output of each sub-layer is LayerNorm(x + Sublayer(x)), where Sublayer(x) is the function implemented by the sub-layer itself.
To facilitate these residual connections, all sub-layers in the model, as well as the embedding layers, produce outputs of dimension 512.
Decoder: The decoder is also composed of a stack of N = 6 identical layers.
In addition to the two sub-layers in each encoder layer, the decoder inserts a third sub-layer, which performs multi-head attention over the output of the encoder stack.
Similar to the encoder, we employ residual connections around each of the sub-layers, followed by layer normalization.
We also modify the self-attention sub-layer in the decoder stack to prevent positions from attending to subsequent positions.
This masking, combined with fact that the output embeddings are offset by one position, ensures that the predictions for position i can depend only on the known outputs at positions less than i.
Attention
An attention function can be described as mapping a query and a set of key-value pairs to an output, where the query, keys, values, and output are all vectors.
The output is computed as a weighted sum of the values, where the weight assigned to each value is computed by a compatibility function of the query with the corresponding key.
"""

print("Original Length:", len(paragraph.split()), "words") # 265 Wörter

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 5.5 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.6.0
    Uninstalling huggingface_hub-1.6.0:
      Successfully uninstalled huggingface_hub-1.6.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


Original Length: 265 words


### Eigene Zusammenfassung vom Text (76 Wörter ~ 100 Tokens)

Encoder: 6 identical layers using self-attention and feed-forward networks to process the input.

Decoder: 6 similar layers that generate output. It adds a third step to attend to the Encoder's data and uses "masking" to prevent looking at future words.

Architecture Rules: Both use residual connections and layer normalization to stabilize data at a fixed dimension of 512.

Attention Mechanism: Calculates output by matching a "query" to "keys," resulting in a weighted sum of relevant "values."

### Zusammenfassung der Parameter

Bevor wir mit Text-Summarization beginnen, ist hier eine kurze Zusammenfassung der Parameter die bei den folgenden Modellen verändert werden können (ein Ausschnitt) und dessen Auswirkung



---


Kontrolle der Sequenzlänge:
* **max_length (Integer):** Setzt die maximale Gesamtlänge der Sequenz (oft inklusive des Eingabetextes, je nach Modell-Pipeline). Das Modell wird abbrechen, sobald dieses Limit erreicht ist (Auch mitten im Satz).

* **max_new_tokens (Integer):** Setzt die maximale Gesamtlänge der neu generierten Token. Das Modell wird abbrechen, sobald dieses Limit erreicht ist.

* **min_length / min_new_tokens (Integer):** Analog zu den Anderen.

* **early_stopping (Boolean/String):** Steuert die Abbruchbedingung für Beam-Search (siehe unten). Akzeptiert drei Werte: Bei True stoppt die Generierung sofort, sobald die gewünschte Anzahl fertiger Satz-Kandidaten gefunden wurde. Bei False wird gestoppt, wenn es mathematisch extrem unwahrscheinlich ist, noch bessere Ergebnisse zu finden. Bei "never" läuft die Suche weiter, bis keine besseren Kandidaten mehr möglich sind.



---


Generierungs-Strategie:
* **do_sample (Boolean):** Aktiviert probabilistisches Sampling. Ist dieses Flag aktiviert, so wählt das Modell nicht immer das wahrscheinlichste Wort (*Greedy Decoding*) als nächstes, sondern ein zufälliges Wort basierend auf den Wahrscheinlichkeiten

* **num_beams (Integer):** Bestimmt die Anzahl der parallel verfolgten Textpfade. Ein Wert von 1 bedeutet, dass das Modell immer nur direkt das nächste Wort wählt. Ein höherer Wert zwingt das Modell, gleichzeitig mehrere Satz-Alternativen im Kopf zu behalten und abzuwägen. Das führt oft zu logischeren und besseren Texten, verlangsamt aber die Generierung.


---


Manipulierung der Logits:
* **temperature (Float):** Steuert den Zufall (Kreativität). Werte nahe 0 machen den Text logisch und vorhersehbar. Werte nahe oder über 1.0 machen den Text kreativer, aber potenziell fehlerhaft.

* **top_k (Integer):** Begrenzt die Auswahl für das nächste Wort strikt auf die k wahrscheinlichsten Wörter. Filtert völlig unpassende Wörter bei hoher Temperatur heraus.

* **top_p (Float):** Das Modell wählt dynamisch nur aus einem Pool von Wörtern, deren addierte Wahrscheinlichkeit p ergibt.

* **repetition_penalty (Float):** Bestraft das Modell für die erneute Verwendung bereits geschriebener Wörter. Ein Wert von 1.0 bedeutet keine Bestrafung, höhere Werte erzwingen einen abwechslungsreicheren Wortschatz und verhindern Endlosschleifen.

* **no_repeat_ngram_size (Integer):** Verhindert Wortschleifen auf Satzebene. Ein Wert von 4 bedeutet, dass keine exakte Abfolge von 4 Tokens im gesamten generierten Text ein zweites Mal vorkommen darf.

## Pegasus = Encoder-Decoder

Pegasus uses 'abstractive' generation. This means, it understands meaning and uses own words for summarization (based on training knowledge).

Pegasus is known to be hallucination-prone because it was trained on BBC articles with punchy and sometimes inaccurate first sentences.

Gap Sentence Generation (GSG): During its initial training, entire "important" sentences were removed from documents. The model had to "fill in the gaps" by generating those missing sentences based on the remaining context. This forced it to learn how to identify and synthesize the most critical information.

Extreme Fine-tuning: The -xsum version is fine-tuned on BBC news articles where the goal is to create a "one-sentence" summary that answers "What is this article about?" rather than just repeating the lead sentence.

#### Strenghts

- True abstraction
- Extreme summarizations

#### Weaknesses

- Hallucinations
- Heavy bias toward producing single sentence outputs
- Relatively large (568 million params)

In [ ]:
from transformers import pipeline

summarizer = pipeline("summarization", model="google/pegasus-xsum")

text = """
The NASA Mars 2020 Perseverance rover is exploring Jezero Crater.
It is looking for signs of ancient life and collecting samples of rock
and regolith (broken rock and soil) for possible return to Earth.
"""

summary = summarizer(text, max_length=60, min_length=10, do_sample=False)

print(summary[0]['summary_text'])

Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-xsum and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use mps:0
Your max_length is set to 60, but your input_length is only 44. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=22)


The US space agency has sent a robot rover to the surface of Mars for the first time.


### Overwhelming the transformer with sequence length and equations

In [ ]:
encode_decode_attention_text = """
3.1 Encoder and Decoder Stacks Encoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two sub-layers. The first is a multi-head self-attention mechanism, and the second is a simple, positionwise fully connected feed-forward network. We employ a residual connection [11] around each of the two sub-layers, followed by layer normalization [1]. That is, the output of each sub-layer is LayerNorm(x + Sublayer(x)), where Sublayer(x) is the function implemented by the sub-layer itself. To facilitate these residual connections, all sub-layers in the model, as well as the embedding layers, produce outputs of dimension dmodel = 512. Decoder: The decoder is also composed of a stack of N = 6 identical layers. In addition to the two sub-layers in each encoder layer, the decoder inserts a third sub-layer, which performs multi-head attention over the output of the encoder stack. Similar to the encoder, we employ residual connections around each of the sub-layers, followed by layer normalization. We also modify the self-attention sub-layer in the decoder stack to prevent positions from attending to subsequent positions. This masking, combined with fact that the output embeddings are offset by one position, ensures that the predictions for position i can depend only on the known outputs at positions less than i.

3.2 Attention An attention function can be described as mapping a query and a set of key-value pairs to an output, where the query, keys, values, and output are all vectors. The output is computed as a weighted sum 3 Scaled Dot-Product Attention Multi-Head Attention Figure 2: (left) Scaled Dot-Product Attention. (right) Multi-Head Attention consists of several attention layers running in parallel. of the values, where the weight assigned to each value is computed by a compatibility function of the query with the corresponding key. 3.2.1 Scaled Dot-Product Attention We call our particular attention "Scaled Dot-Product Attention" (Figure 2). The input consists of queries and keys of dimension dk, and values of dimension dv. We compute the dot products of the query with all keys, divide each by √ dk, and apply a softmax function to obtain the weights on the values. In practice, we compute the attention function on a set of queries simultaneously, packed together into a matrix Q. The keys and values are also packed together into matrices K and V . We compute the matrix of outputs as: Attention(Q, K, V ) = softmax(QKT √ dk )V (1) The two most commonly used attention functions are additive attention [2], and dot-product (multiplicative) attention. Dot-product attention is identical to our algorithm, except for the scaling factor of √ 1 dk . Additive attention computes the compatibility function using a feed-forward network with a single hidden layer. While the two are similar in theoretical complexity, dot-product attention is much faster and more space-efficient in practice, since it can be implemented using highly optimized matrix multiplication code. While for small values of dk the two mechanisms perform similarly, additive attention outperforms dot product attention without scaling for larger values of dk [3]. We suspect that for large values of dk, the dot products grow large in magnitude, pushing the softmax function into regions where it has extremely small gradients 4 . To counteract this effect, we scale the dot products by √ 1 dk . 3.2.2 Multi-Head Attention Instead of performing a single attention function with dmodel-dimensional keys, values and queries, we found it beneficial to linearly project the queries, keys and values h times with different, learned linear projections to dk, dk and dv dimensions, respectively. On each of these projected versions of queries, keys and values we then perform the attention function in parallel, yielding dv-dimensional 4To illustrate why the dot products get large, assume that the components of q and k are independent random variables with mean 0 and variance 1. Then their dot product, q · k = Pdk i=1 qiki, has mean 0 and variance dk. 4 output values. These are concatenated and once again projected, resulting in the final values, as depicted in Figure 2. Multi-head attention allows the model to jointly attend to information from different representation subspaces at different positions. With a single attention head, averaging inhibits this. MultiHead(Q, K, V ) = Concat(head1, ..., headh)WO where headi = Attention(QWQ i , KW K i , V WV i ) Where the projections are parameter matrices W Q i ∈ R dmodel×dk , W K i ∈ R dmodel×dk , WV i ∈ R dmodel×dv and WO ∈ R hdv×dmodel . In this work we employ h = 8 parallel attention layers, or heads. For each of these we use dk = dv = dmodel/h = 64. Due to the reduced dimension of each head, the total computational cost is similar to that of single-head attention with full dimensionality. 3.2.3 Applications of Attention in our Model The Transformer uses multi-head attention in three different ways: • In "encoder-decoder attention" layers, the queries come from the previous decoder layer, and the memory keys and values come from the output of the encoder. This allows every position in the decoder to attend over all positions in the input sequence. This mimics the typical encoder-decoder attention mechanisms in sequence-to-sequence models such as [38, 2, 9]. • The encoder contains self-attention layers. In a self-attention layer all of the keys, values and queries come from the same place, in this case, the output of the previous layer in the encoder. Each position in the encoder can attend to all positions in the previous layer of the encoder. • Similarly, self-attention layers in the decoder allow each position in the decoder to attend to all positions in the decoder up to and including that position. We need to prevent leftward information flow in the decoder to preserve the auto-regressive property. We implement this inside of scaled dot-product attention by masking out (setting to −∞) all values in the input of the softmax which correspond to illegal connections. See Figure 2.
"""

In [ ]:
summary = summarizer(encode_decode_attention_text, max_length=60, min_length=10, do_sample=False)
print(summary[0]['summary_text'])

Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-xsum and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use mps:0
Token indices sequence length is longer than the specified maximum sequence length for this model (1331 > 512). Running this sequence through the model will result in indexing errors


In this paper, we present a novel multi-head self-attention encoder and decoder.


In [ ]:
summary = summarizer(encode_decode_attention_text, max_length=60, min_length=10, do_sample=False, truncation=True)
print(summary[0]['summary_text'])

Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-xsum and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use mps:0


In this paper, we describe two encoders and two decoders, each with its own self-attention mechanism.


## BART-large-cnn (0.4B params)

Hugging Face link: https://huggingface.co/facebook/bart-large-cnn

**Bart** ist ein **Encoder-Decoder** Modell und daher besonders gut geeignet (gemäß Vorlesungs-Slides) für Text-Zusammenfassungen, aber auch beispielsweise Übersetzungen. Diese Version von BART (BART selbst wurde auf "allgemeinen" Text trainiert) wurde auf Texten von *CNNN Daily Mail* fine-getuned.

In [ ]:
bart = pipeline("summarization", model="facebook/bart-large-cnn")

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cpu


In [ ]:
summary = bart(paragraph)
print(summary[0]['summary_text'])

Encoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two sub-layers. The first is a multi-head self-attention mechanism, and the second is a simple, positionwise fully connected feed-forward network. The decoder inserts a third sub-layer which performs multi- head attention over the output of the encoder stack.


Unterscheidung zwischen Encoder und Decoder und layers. Keine Erklärung von Masking, Attention und Residual Connections

In [ ]:
summary = bart(paragraph, min_length=100)
print(summary[0]['summary_text'])

Encoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two sub-layers. The first is a multi-head self-attention mechanism, and the second is a simple, positionwise fully connected feed-forward network. The decoder inserts a third sub-layer, which performsMulti-head attention over the output of the encoder stack. The output is computed as a weighted sum of the values, where the weight assigned to each value is computed by a compatibility function.


100 Tokens sind ca. die Länge von der eigenen Zusammenfassung. Text wurde länger, aber weiterhin keine Erwähnung von den oben gennanten Bereichen.

In [ ]:
summary = bart(
    paragraph,
    min_length=100,
    do_sample=True,
    temperature=0.8,
    repetition_penalty=2.5,
    no_repeat_ngram_size=3
)

print(summary[0]['summary_text'])

Encoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two sub-layers. The first is a multi-head self-attention mechanism, and the second is a simple, positionwise fully connected feed-forward network. Decoder: In addition to the twoSubLayers in each encoder layer, the decoder inserts a third sub-layer, which performs multi- head attention over the output of the encoder stack. This ensures that predictions for position i can depend only on the known outputs at positions less than i.


Hier wurde versucht das Modell "kreativer" werden zu lassen indem Sampling eingeschaltet wird mit hoher Temperatur. Die Repetition Penalty und der letzte Parameter soll wiederholungen verhindern. Text ist sehr ähnlich wie der vorher, nur wurde gegen Ende Halluziniert

In [ ]:
summary = bart(
    paragraph,
    max_length=150,
    min_length=100,
    do_sample=True,
    temperature=1,
    repetition_penalty=2.5,
    no_repeat_ngram_size=4,
    top_k=30,
    top_p=0.9
)

print(summary[0]['summary_text'])

Encoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two sub-layers. First is a multi-head self-attention mechanism, and the second is a simple, positionwise fully connected feed-forward network. Decoder: The decoder is also composed of astack of N =6 identical layers. It inserts a third sub-layer, which performs multi-head attention over the output of the encoder stack. We employ residual connections around each of the sub-layer, followed by layer normalization.


Da der Text wieder sehr viel copy and paste vom originalen Text ist wurde hier versucht die Kreativität weiter zu erhöhen. Dennoch kein großer Unterschied zum vorherigen Text.

Wie oben erwähnt wurde **BART** auf News Artikeln trainiert, wo vermutlich immer die relevanteste Information zu Beginn steht und daher übernimmt das Modell oft den Anfang.

## T5-Small

In [ ]:
summarizer = pipeline("summarization", model="t5-small")

summary = summarizer(
    "summarize: " + text,
    max_length=150,
    min_length=60,
    num_beams=6,
    length_penalty=1.5,
    repetition_penalty=1.3,
    no_repeat_ngram_size=3,
    do_sample=False,
    truncation=True
)

print("Model Summary:\n")
print(summary[0]["summary_text"])

True


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Device set to use cuda:0
Both `max_new_tokens` (=256) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Model Summary:

the encoder is composed of a stack of N = 6 identical layers . each layer has two sub-layers, the first is a multi-head self-attention mechanism . the decoder inserts a third in each encoder layer to prevent positions from attending to subsequent positions .


Die Ergebnisse der Zusammenfassung zeigen, dass T5-small einige zentrale Konzepte aus dem Transformer-Abschnitt erkennen kann, aber Schwierigkeiten hat, dichten technischen Text vollständig und zusammenhängend zusammenzufassen. In der Version ohne Chunking erzeugte das Modell nur eine sehr kurze Summary, die zwar einige wichtige Ideen wie die Encoder-Struktur mit sechs Schichten, Layer Normalization und Scaled Dot-Product Attention erwähnt, aber viele wesentliche Aspekte auslässt. Dazu gehören unter anderem das Masking im Decoder, Residual Connections, die Rolle von Queries, Keys und Values sowie die verschiedenen Einsatzarten von Multi-Head Attention im Transformer.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

encoder_decoder_chunk, attention_rest = paragraph.split("An attention function can be described as", 1)

chunk1 = encoder_decoder_chunk.strip()
chunk2 = ("An attention function can be described as" + attention_rest).strip()

chunks = [chunk1, chunk2]

def summarize_text(input_text, max_new_tokens=110, min_length=50):
    prompt = "summarize the following transformer architecture text clearly and completely: " + input_text

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    summary_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        min_length=min_length,
        num_beams=10,
        length_penalty=1.0,
        repetition_penalty=1.5,
        no_repeat_ngram_size=3,
        early_stopping=True
    )

    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)


chunk_summaries = []

for i, chunk in enumerate(chunks, 1):
    summary = summarize_text(chunk, max_new_tokens=110, min_length=50)
    chunk_summaries.append(summary)

    print(f"\nSummary of chunk {i}:")
    print(summary)

combined_summary = " ".join(chunk_summaries)

print("\nCombined Summary:\n")
print(combined_summary)

final_summary = summarize_text(combined_summary, max_new_tokens=170, min_length=100)

print("\nFinal Summary:\n")
print(final_summary)

True

Summary of chunk 1:
the encoder is composed of a stack of N = 6 identical layers. each layer has two sub-layers. the first is a multi-head self-attention mechanism, and the second is an integrated feed-forward network.

Summary of chunk 2:
an attention function can be described as mapping a query and key-value pairs to an output, where the query, keys, values, and output are all vectors. the output is computed as a weighted sum of the values.

Combined Summary:

the encoder is composed of a stack of N = 6 identical layers. each layer has two sub-layers. the first is a multi-head self-attention mechanism, and the second is an integrated feed-forward network. an attention function can be described as mapping a query and key-value pairs to an output, where the query, keys, values, and output are all vectors. the output is computed as a weighted sum of the values.

Final Summary:

the encoder is composed of a stack of N = 6 identical layers. each layer has two sub-layers. the first i

Der Chunking-Ansatz hat das Ergebnis deutlich verbessert. Durch die Aufteilung des Textes in thematische Abschnitte konnte das Modell mehr Informationen zu Attention, additive versus dot-product attention und Multi-Head Attention erhalten. Das zeigt, dass Chunking eine sinnvolle Methode ist, wenn der Eingabetext für ein kleines Modell zu lang oder zu komplex ist. Trotzdem verlor auch die abschließende, automatisch erzeugte Gesamtsummary wieder wichtige Informationen und enthielt teils ungenaue Formulierungen wie „integrated feed-forward network“, was nicht der Terminologie des Originaltexts entspricht.


Im Vergleich zur manuellen Zusammenfassung waren die Modellzusammenfassungen insgesamt ungenauer, unvollständiger und weniger gut strukturiert. Die manuelle Summary konnte die Beziehung zwischen Encoder-Decoder-Architektur, Scaled Dot-Product Attention und dem Masking im Decoder deutlich besser darstellen. Insgesamt zeigt dieses Experiment, dass T5-small zwar Kerngedanken erkennen kann, aber akademischen Text oft zu stark komprimiert und wichtige technische Details verliert. Die Chunking-Version war klar besser als die Zusammenfassung ohne Chunking und daher die stärkere Modelllösung.

## FLAN-T5-large (0.8B params)

Hugging Face link: https://huggingface.co/google/flan-t5-large

FLAN-T5 ist (ebenso wie BART) ein Encoder-Decoder Modell. Es basiert auf der T5-Architektur (Text-to-Text Transfer Transformer), bei der prinzipiell jede NLP-Aufgabe als "Text-in, Text-out" Problem formuliert wird. Das Besondere an der "FLAN"-Version ist, dass das Basismodell massiv durch Instruction Tuning auf tausenden verschiedenen Datensätzen fine-getuned wurde. Das bedeutet, es wurde speziell darauf trainiert, direkte Anweisungen im Text zu befolgen, weshalb es auch bei neuen, ungesehenen Aufgaben (Zero-Shot) oft gut abschneidet.

In [ ]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

t5_tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-large")
t5_model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-large")

prompt = f"Briefly summarize the following text:\n\n{paragraph}"
input_ids = t5_tokenizer(prompt, return_tensors="pt").input_ids

In [ ]:
outputs = t5_model.generate(input_ids)
print(t5_tokenizer.decode(outputs[0]))

<pad>A multi-head attention model for a multi-layer encoder and decoder stack.


Hier ist der Text sehr kurz und sinnlos und es wurden interne Tokens mit übergeben. Dadurch versuchen wir den Text länger und kreativer zu gestalten

In [ ]:
outputs = t5_model.generate(
    input_ids,
    min_new_tokens=100,
    max_new_tokens=150,
    do_sample=True,
    temperature=0.7,
    top_p=0.9
)

print(t5_tokenizer.decode(outputs[0]))

<pad> A model for multi-head attention based decoders and encoders. The model is based on two sub-layers. The encoders encode and decode the inputs of a multi-head attention model. The decoders decode the outputs of the encoder model. The model is based on two sub-layers. The encoders encode the inputs of a multi-head attention model and decode the outputs of a multi-head attention model. The model is based on two sub-layers.</s>


Durch die Kreativität und der erhöhten Länge fängt das Model an sich zu wiederholen. Da das Modell den Prompt sehr genau folgt, versuchen wir hier den Prompt anzupassen und die Kreativität wieder zu senken mit Beam-Search, sowie Wiederholungen zu vermeiden.

In [ ]:
prompt = f"Summarize the following text into short bullet points focusing on Encoder, Decoder, Architecture, and Attention: \n\n{paragraph}"
input_ids = t5_tokenizer(prompt, return_tensors="pt").input_ids

outputs = t5_model.generate(
    input_ids,
    max_new_tokens=150,
    num_beams=5,
    repetition_penalty=1.8,
    no_repeat_ngram_size=3,
    early_stopping=True
)

print(t5_tokenizer.decode(outputs[0], skip_special_tokens=True))

A multi-head self-attention model and a simple, positionwise fully connected feed-forward network.


Hier ist der Output wieder wesentlich zu wenig und nicht sinnvoll

In [ ]:
outputs = t5_model.generate(
    input_ids,
    min_new_tokens=50,
    max_new_tokens=150,
    num_beams=4,
    length_penalty=1.5,
    repetition_penalty=1.2,
    no_repeat_ngram_size=4,
    early_stopping=False
)
print(t5_tokenizer.decode(outputs[0], skip_special_tokens=True))

A multi-head self-attention model and a simple, positionwise fully connected feed-forward network are used to encode and decode the outputs of the encoder stack. A third sub-layer performs multi-head attention over the output of the decoder stack.


Auch das Erzwingen von mehr output und logischen Sätzen liefert kein besseres Ergebnis.

## Qwen2.5-1.5B-Instruct (1.5B params)

Hugging Face link: https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct

Qwen2.5 ist ein reines Decoder-only Modell. Es nutzt die klassische autoregressive Transformer-Architektur, die heutzutage den Standard für große Sprachmodelle (LLMs) wie GPT-4 oder Llama bildet. Das bedeutet, es liest den bisherigen Text und sagt immer nur das wahrscheinlichste nächste Wort voraus.

Das Besondere an dieser "Instruct"-Version (entwickelt von Alibaba Cloud) ist, dass sie nach dem anfänglichen Training massiv für das Befolgen komplexer Anweisungen und das Führen natürlicher Dialoge fine-getuned wurde (Instruction Tuning & RLHF). Qwen ist zwar laut "Lehrbuch" nicht am Besten für Text-Zusammenfassungen geeignet, wir werden es aber dennoch hier versuchen, da es mehr Parameter als die anderen Modelle besitzt und deswegen eventuell leistungsstärker ist.

In [ ]:
qwen_generator = pipeline("text-generation", model="Qwen/Qwen2.5-1.5B-Instruct", device_map="auto")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cpu


In [ ]:
prompt = f"Briefly summarize me the following paragraph: {paragraph}"
messages = [
    {"role": "system", "content": "You are an expert AI assistant. Your job is to concisely summarize complex machine learning architectures, capturing the core technical concepts and discarding unnecessary details."},
    {"role": "user", "content": prompt}
]

output = qwen_generator(
    messages,
    max_new_tokens=300,
    do_sample=True,
    temperature=0.2,
    top_k=20,
    top_p=0.90,
    repetition_penalty=1.1
)

print(output[0]['generated_text'][-1]['content'])

**Summary:**

The encoder-decoder architecture consists of multiple layers, each containing a multi-head self-attention mechanism and a feed-forward neural network. Residual connections and layer normalization are used throughout. The decoder includes additional multi-head attention over the encoder's output, preventing future positions from influencing current ones through masking techniques.


Diese Zusammenfassung ist schon ziemlich akkurat. Allerdings wurde Masking falsch erklärt und der letzte Absatz zu Attention nicht erwähnt. Dadurch vesuchen wir die Kreativität etwas anzukurbeln


In [ ]:
output = qwen_generator(
    messages,
    max_new_tokens=300,
    do_sample=True,
    temperature=0.4,
    top_k=50,
    top_p=0.9,
    repetition_penalty=1.1
)

print(output[0]['generated_text'][-1]['content'])

**Summary:**

The transformer architecture consists of an encoder and a decoder. Both use multiple layers with self-attention mechanisms and feed-forward networks. Residual connections and layer normalization are applied around each layer. The decoder includes additional multi-head attention over the encoder's output. Masking prevents future context from affecting current predictions. This setup allows efficient parallel processing across sequences.


Diese Antwort hier ist ziemlich gut

# Q&A

## bert-large-uncased-whole-word-masking-finetuned-squad

In [ ]:
qa_pipeline = pipeline(
    "question-answering",
    model="bert-large-uncased-whole-word-masking-finetuned-squad"
)

question = "How is the output of attention computed?"
result = qa_pipeline(
    question=question,
    context=text
)

print("Answer:", result["answer"])
print("Score:", round(result["score"], 4))
print("Question:", question)

True


Some weights of the model checkpoint at bert-large-uncased-whole-word-masking-finetuned-squad were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cuda:0


Answer: as a weighted sum
of the values
Score: 0.9948
Question: How is the output of attention computed?


Das verwendete QA-Modell bert-large-uncased-whole-word-masking-finetuned-squad funktionierte gut bei faktischen Fragen, deren Antworten klar und direkt im Text enthalten waren. Zum Beispiel wurde die Frage „How many identical layers does the encoder have?“ korrekt mit „N = 6“ beantwortet, bei einem hohen Confidence Score von 0.8941. Das zeigt, dass das Modell gut darin ist, kurze und exakte Antwortspannen aus dem gegebenen Kontext zu extrahieren.

Besonders gut funktionierte das Modell bei Fragen, die sich auf klar identifizierbare Textstellen beziehen, etwa zur Anzahl der Schichten, zur ersten Sub-Layer des Encoders oder zur Grunddefinition von Attention. In solchen Fällen konnte das Modell die richtige Antwort mit guter Genauigkeit finden. Schwächer war die Leistung jedoch bei längeren oder technisch komplexeren Fragen, insbesondere wenn der Kontext sehr lang war oder die richtige Antwort eine längere Erklärung erforderte. In einem Fall wurde zum Beispiel nur „vectors“ als Antwort gegeben, obwohl die eigentliche Antwort deutlich länger und spezifischer war. Das zeigt, dass der Score allein nicht ausreicht und jede Antwort zusätzlich inhaltlich geprüft werden muss.

Insgesamt war das QA-Modell gut geeignet, um klare Fakten aus dem Text herauszuziehen. Seine Hauptgrenze lag darin, dass es bei komplexeren technischen Fragen manchmal nur unvollständige Antwortspannen auswählte. Daraus lässt sich schließen, dass das Modell für präzise Faktenextraktion gut funktioniert, aber bei tieferem technischen Verständnis oder längeren Erklärungen nur eingeschränkt zuverlässig ist, wenn der Kontext nicht gezielt eingegrenzt wird.

## RoBERTa-base-squad2 (125M params)

Hugging Face link: https://huggingface.co/deepset/roberta-base-squad2

RoBERTa ist  ein reines Encoder-Modell. Es baut auf der bekannten BERT-Architektur auf, wurde aber deutlich robuster und effizienter trainiert (mehr Daten, längeres Training, dynamisches Maskieren von Wörtern). Diese spezifische Version wurde auf dem SQuAD 2.0 Datensatz (Stanford Question Answering Dataset) fine-getuned. Das macht dieses Modell zu einem absoluten Spezialisten für Extractive Question Answering: Man gibt dem Modell einen Text (Kontext) und eine Frage dazu. Das Modell generiert dann keinen neuen Text, sondern extrahiert genau die Textstelle, die die Antwort enthält – oder erkennt zuverlässig, wenn die Antwort gar nicht im Text zu finden ist.

In [ ]:
qa_model = pipeline("question-answering", model="deepset/roberta-base-squad2")

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/496M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/79.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

Device set to use cpu


In [ ]:
questions = [
    "How many identical layers is the encoder composed of?",
    "What is the output dimension of the sub-layers?",
    "What does the decoder insert that the encoder does not have?",
    "What is an encoder?",
    "What is a decoder?",
    "How can an attention function be described?",
    "Are cats cute?"
]

for q in questions:
    result = qa_model(question=q, context=paragraph)
    print(f"Question: {q}")
    print(f"Answer: '{result['answer']}'")
    print(f"Confidence Score: {round(result['score'], 4)}\n")

Question: How many identical layers is the encoder composed of?
Answer: 'N = 6'
Confidence Score: 0.5962

Question: What is the output dimension of the sub-layers?
Answer: '512'
Confidence Score: 0.3522

Question: What does the decoder insert that the encoder does not have?
Answer: 'a third sub-layer'
Confidence Score: 0.0002

Question: What is an encoder?
Answer: 'a stack of N = 6 identical layers'
Confidence Score: 0.2848

Question: What is a decoder?
Answer: 'a stack of N = 6 identical layers'
Confidence Score: 0.1791

Question: How can an attention function be described?
Answer: 'mapping a query and a set of key-value pairs to an output'
Confidence Score: 1.1653

Question: Are cats cute?
Answer: '
Encoder'
Confidence Score: 0.0



Es gibt hier nicht wirklich Parameter die man tunen könnte, da keine Antwort generiert wird, sondern nur die Text-Positionen zwischen denen sich die Antwort im original Text befindet vorhergesagt wird.

Prinzipiell sind die Antworten nicht so schlecht, wenn man spezifische Fragen stellt (beispielsweise mit den Dimensionen), was auf ein gutes Textverständnis hinweist. Allerdings sind vor allem die Fragen "What is an ..?" nicht sehr gut beantwortet und auch die dritte Frage liefert eine ziemlich niedrige Confidence obwohl sie mehr oder weniger richtig ist.

In der Standardeinstellung werden Fragen nicht beantwortet, die nicht im Text vorkommen, daher liefert das Modell keine Antwort und eine Confidence von 0 auf die letzte Frage


In [ ]:
result = qa_model(question="Are cats cute?", context=paragraph, handle_impossible_answer=True)
print(f"Question: {q}")
print(f"Answer: '{result['answer']}'")
print(f"Confidence Score: {round(result['score'], 4)}\n")

Question: Are cats cute?
Answer: ''
Confidence Score: 0.9211



Interessanterweise liefert das Model ebenfalls keine Antwort (was richtig ist) wenn man die Beantwortung von "impossible questions" aktiviert und das mit einer sehr hohen Confidence

## Distilbert = Encoder only

Smaller, faster version of BERT (Bidirectional Encoder Representations from Transformers)

A BERT model was used to distill a smaller model out of it.

#### BERT = Encoder Decoder

Is able to read whole sentences simultaneously.
Hiding words in training texts (= masking) was used in training).

Sentiment Analysis: Is this review angry or happy?
Named Entity Recognition (NER): Which words are names of people or cities?
Question Answering: Finding the exact sentence in a paragraph that answers a user's question.

#### Strengths

- Fast and efficient

#### Weaknesses

- No 'Next Sentence Prediction'
- Not generative

### Halucinations

In [ ]:
from transformers import DistilBertTokenizer, DistilBertModel
import torch
question_answerer = pipeline("question-answering", model='distilbert-base-cased-distilled-squad')
result = question_answerer(question="What was the research objective?", context=encode_decode_attention_text_stripped)
print(result)

Device set to use mps:0


{'score': 0.10681694000959396, 'start': 4851, 'end': 4888, 'answer': 'preserve the auto-regressive property'}


In [ ]:
question_answerer = pipeline("question-answering", model='distilbert-base-cased-distilled-squad')
result = question_answerer(question="How much is one plus one?", context=encode_decode_attention_text_stripped)
print(result)

Device set to use mps:0


{'score': 0.011308356188237667, 'start': 4408, 'end': 4468, 'answer': 'all of the keys, values and queries come from the same place'}


### Meaningful questions

In [ ]:
question_answerer = pipeline("question-answering", model='distilbert-base-cased-distilled-squad')
result = question_answerer(question="What is the name of the particular attention function used?", context=encode_decode_attention_text_stripped)
print(result)

Device set to use mps:0


{'score': 0.8382486999034882, 'start': 1705, 'end': 1733, 'answer': 'Scaled Dot-Product Attention'}


In [ ]:
question_answerer = pipeline("question-answering", model='distilbert-base-cased-distilled-squad')
result = question_answerer(question="What model architecture uses multi-head attention in three different ways?", context=encode_decode_attention_text_stripped)
print(result)

Device set to use mps:0


{'score': 0.8213588361104485, 'start': 3919, 'end': 3934, 'answer': 'The Transformer'}


# Text Generation for Research Paper Expansion

Hier werden wir den Absatz für die Erklärung vom Encoder kopieren und versuchen diesen in einfachen Worten zu erklären, für Personen die kein Domänenwissen besitzen

In [ ]:
paper_part = '''"Encoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two sub-layers.
The first is a multi-head self-attention mechanism, and the second is a simple, positionwise fully connected feed-forward network.
We employ a residual connection around each of the two sub-layers, followed by layer normalization.
That is, the output of each sub-layer is LayerNorm(x + Sublayer(x)), where Sublayer(x) is the function implemented by the sub-layer itself.
To facilitate these residual connections, all sub-layers in the model, as well as the embedding layers, produce outputs of dimension 512."'''

prompt = paper_part + "\nTL;DR:\n"

print("Länger der Sektion vom Paper: {}".format(len(paper_part.split(' '))))

def truncate_prompt_and_print_output(output, prompt_text):
    full_text = output[0]['generated_text']
    new_text = full_text.replace(prompt_text, "").strip()
    print(new_text)

Länger der Sektion vom Paper: 90


## FLAN-T5 = Encoder-Decoder

Thus, FLAN-T5 can understand and write.

FLAN (Fine-tuned LAnguage Net): Instead of just learning from raw text, the model was "instruction-tuned"

T5 (Text-to-Text Transfer Transformer): T5 treats every NLP problem as a "text-to-text" task. Whether it's classification or translation, the input is text and the output is always a new string of text.

"echoing" is a known quirk of T5

In [ ]:
generator = pipeline("text2text-generation", model="google/flan-t5-base")
response = generator("Why are bananas curved?", max_length=100)
print(response[0]['generated_text'])

Device set to use mps:0
Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


they are curved to make them easier to eat.


In [ ]:
response = generator("Explain the scientific biological reason why bananas are curved.", max_length=100)
print(response[0]['generated_text'])

Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


curved bananas are a type of fruit.


In [ ]:
response = generator("What is a transformer architecture in machine learning", max_length=100)
print(response[0]['generated_text'])

Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A transformer architecture is a machine learning architecture that combines a machine learning architecture with a machine learning architecture that combines a machine learning architecture with a machine learning architecture that combines a machine learning architecture with a machine learning architecture that combines a machine learning architecture with a machine learning architecture that combines a machine learning architecture with a machine learning architecture that combines a machine learning architecture with a machine learning architecture that combines a machine learning architecture with a machine learning architecture that combines a machine learning architecture with a machine learning architecture that combines a machine learning architecture with a machine learning architecture that combines a machine learning architecture with a machine learning architecture that combines a machine learning architecture with a machine learning architecture that combines a machine l

In [ ]:
response = generator("What is a transformer architecture in machine learning?", max_length=100)
print(response[0]['generated_text'])

Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


a neural network


In [ ]:
response = generator("How much is one plus one", max_length=100)
print(response[0]['generated_text'])

Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


one plus one is one plus one.


### Flan-T5 strengths (sentiment analysis and translations)

In [ ]:
response = generator("Review: This movie was a complete waste of time. Is this review positive or negative?", max_length=100)
print(response[0]['generated_text'])

Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


negative


In [ ]:
response = generator("Translate to Spanish: Where is the nearest library?", max_length=100)
print(response[0]['generated_text'])

Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Dónde está la biblioteca más cercana?


### FLAN-T5 research expansion

In [ ]:
response = generator("""Input: Explain the Multi-Head Attention layer.
Output: It allows the model to focus on different parts of the input sequence simultaneously by using multiple sets of weights.
Input: Explain the Position-wise Feed-Forward layer.
Output:""", max_new_tokens=100,
    repetition_penalty=1.2,
    do_sample=True,
    temperature=0.7)
print(response[0]['generated_text'])

It allows the model to focus on different parts of the input sequence simultaneously by using multiple sets of weights.


## GPT-2 (0.8B params)

Hugging Face link: https://huggingface.co/openai-community/gpt2-large

GPT-2 ist ein reines Decoder-only Modell. Es arbeitet autoregressiv, was bedeutet, dass es Text Wort für Wort von links nach rechts generiert, wobei jedes neue Wort auf den vorherigen basiert.

Es war das erste Modell, das demonstrierte, dass ein Transformer durch reines Training auf gewaltigen Textmengen (WebText) fähig ist, verschiedene Aufgaben wie Zusammenfassungen oder Übersetzungen ohne explizites Training für diese Aufgabe zu lösen (Zero-Shot-Fähigkeit). In der Architektur wird in jedem Layer Masked Self-Attention verwendet, um sicherzustellen, dass das Modell beim Training nicht auf nachfolgende Wörter zugreifen kann.

In [ ]:
gpt2 = pipeline("text-generation", model="gpt2-large")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.25G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cpu


In [ ]:
output = gpt2(
    prompt,
    max_new_tokens=100,
    do_sample=False
)
truncate_prompt_and_print_output(output, prompt)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


The model is composed of 6 identical layers, each with two sub-layers.
The first is a multi-head self-attention mechanism, and the second is a simple, positionwise fully connected feed-forward network.
The residual connections are used to facilitate residual connections between the sub-layers.
The model is trained using a gradient descent algorithm.
The model is trained using a gradient descent algorithm.
The model is trained using a gradient descent algorithm.
The model


Hier liefert das Modell keine Gute Erklärung sondern wiederholte einfach nur Tokens aus dem Input, danach beginnt es zu Halluzinieren und wiederholt den gleichen Satz. Als kleinen Versuch setzen wir hier die Temperatur hoch und geben eine Penalty für Wiederholungen

In [ ]:
output = gpt2(
    prompt,
    max_new_tokens=150,
    do_sample=True,
    temperature=1.0,
    repetition_penalty=1.2
)
truncate_prompt_and_print_output(output, prompt)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


1) Build a 3D binary gradient descent neural network, i.e.: In order to obtain deep learning for Image segmentation, you must implement an convolutional neural networks.  A network can be shown above as a rectangular manifold (the hidden layer can have any number from 0 : 11, so it doesn't matter which columns are labelled). Note that if we try to get a 1-based output, we will end with no visible information since there's only one column labeled "0". Instead, we need to divide that input value into 4 'hidden' segments, or alternatively use 4 'outputs'. And of course 4 inputs can appear directly behind each other on the screen! This allows us to achieve our desired level of deep


Wie zu erwarten hat hier das Modell stark Halluziniert und einfach irgendeinen Output generiert (der aber faiererweise zum Themenbereich halbwegs passt). Als nächstes versuchen wir die Temperatur zu drosseln um fachliche Antworten zu erzwingen

In [ ]:
output = gpt2(
    prompt,
    max_new_tokens=150,
    do_sample=True,
    temperature=0.01,
    repetition_penalty=1.1
)
truncate_prompt_and_print_output(output, prompt)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


The model is composed of 6 identical layers, each with 2 sub-layers.
Each sub-layer produces an output of size 512.
The residual connections between the sub-layers are used to connect the output of each sub-layer to the input of the next.
The residual connections between the embedding layers are used to connect the output of each sub-layer to the input of the next.
The residual connections between the model's input and output layers are used to connect the output of each sub-layer to the input of the next.
The residual connections between the model's input and output layers are used to connect the output of each sub-layer to the input of the next.
The residual connections between the


Das ist keine Erklärung in einfachen Worten und das Model wiederholt sich wieder. Daher versuchen wir hier die Kreativität zu erhöhen und Wiederholungen zu bestrafen.

In [ ]:
output = gpt2(
    prompt,
    max_new_tokens=150,
    do_sample=True,
    temperature=0.3,
    top_k=30,
    repetition_penalty=1.8,
    no_repeat_ngram_size=3
)
truncate_prompt_and_print_output(output, prompt)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


1 - A multilayer perceptron with an embedded recurrent neural net (RNN). 2 – An RNG that generates random values for x. 3 – Feed forward networks are used to generate input from this hidden state space using gradient descent 4 -- Residual connectivity between different nodes allows them to learn about their own inputs 5 – Stochastic Gradient Descent algorithm uses loss functions based on previous learning results 8/9


Leider ist auch dieser Output Schlecht und daher versuchen wir ein besseres Modell.

## Qwen2.5-1.5B-Instruct (1.5B params)

Hugging Face link: https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct

GPT-2-L stammt aus dem Jahr 2019, bestitz weniger Parameter und wurde nicht durch Instruction Tuning oder RLHF darauf trainiert, Befehle zu befolgen. Wir nutzen es hier, um zu demonstrieren dass ein modernes 1.5B Modell (Qwen) dieses ältere Modell in der Logik und Textqualität vermutlich weit übertreffen wird.


In [ ]:
qwen_generator = pipeline("text-generation", model="Qwen/Qwen2.5-1.5B-Instruct", device_map="auto")

Device set to use cpu


In [ ]:
prompt = "Explain this to me in simple language, so that I understand it without domain knowledge: \n\n" + paper_part
messages = [
    {"role": "system", "content": "You are a helpful AI tutor. Your job is to explain complex machine learning text in simple language for beginners."},
    {"role": "user", "content": prompt}
]

In [ ]:
output = qwen_generator(
    messages,
    max_new_tokens=150,
    do_sample=True,
    temperature=0.7,
    top_p=0.90,
    repetition_penalty=1.1
)

print(final_text = output[0]['generated_text'][-1]['content'])


--- AI EXPLANATION ---
Sure! Let's break down what an "encoder" does in simpler terms:

Imagine you have a big puzzle with many pieces. To solve it, you need to organize these pieces into smaller groups before putting them back together. That's kind of like how an encoder works!

In our case:
- **N = 6**: This means we use six different ways to organize or process information (like having six puzzles).
- **Self-Attention Mechanism**: Think of this as a special way to see how things fit together within your organization. It helps the encoder understand which parts work best together.
- **Position-wise Fully Connected Feed-Forward Networks**: These networks help the encoder understand where everything is placed in space, just like knowing your place on


Wir haben bei diesem Durchgang das Qwen-Modell mit kreativen Einstellungen genutzt, um eine vereinfachte Analogie für den Encoder zu erzeugen.

Nicht gut an der Zusammenfassung ist, dass wir das Token-Limit mit max_new_tokens=150 zu niedrig angesetzt haben, wodurch die Erklärung mitten im Satz abbricht, und dass die gewählte Puzzle-Metapher zwar anschaulich ist, aber die technischen Details der Architektur eher umschreibt als präzise zusammenzufasst.

In [ ]:
output = qwen_generator(
    messages,
    max_new_tokens=300,
    do_sample=True,
    temperature=0.2,
    top_k=20,
    top_p=0.90,
    repetition_penalty=1.1
)

print(output[0]['generated_text'][-1]['content'])

Sure! Let's break down this concept into simpler terms:

Imagine you have a big box (we'll call it an "encoder") that needs to process information. This box has multiple smaller boxes inside it (these are our "layers"). Each small box does something specific with the information.

Here’s what happens step-by-step:

1. **Self-Attention Mechanism**: Think of this like a smart assistant who can focus on different parts of the information at once. It looks at how important each part is compared to others.

2. **Position-wise Fully Connected Feed-Forward Network**: This is like a helper that processes the information based on its position within the sequence. For example, if we're talking about words in a sentence, this helper might look at how similar or different certain words are from their neighbors.

3. **Residual Connections**: Imagine you start with some information (let's say it's your name), and then you add more information through these smaller boxes. If one box adds too much, yo

Wir haben in diesem Versuch mit einer sehr niedrigen Temperatur und einem erhöhten Token-Limit verwendet, um eine schrittweise Erklärung der Komponenten zu erzwingen.

Nicht gut an der Zusammenfassung ist, dass trotz des höheren Limits (max_new_tokens=300) die Erklärung am Ende erneut abrupt abbricht, und dass die fachliche Korrektheit der Analogien, insbesondere bei den Residual Connections und der Position-Wise-Feed-Forward-Neural-Network, zugunsten der Einfachheit stark verzerrt wurde.



In [ ]:
output = qwen_generator(
    messages,
    max_new_tokens=300,
    do_sample=False,
    repetition_penalty=1.05
)

print(output[0]['generated_text'][-1]['content'])

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Sure! Let's break down this concept into simpler terms:

Imagine you have a big box (we'll call it an "encoder") that needs to process information. This box has several smaller boxes inside it (these are like the "layers" in our model).

Each of these smaller boxes has two parts:
1. **Self-Attention Mechanism**: This part helps the box understand how different pieces of information relate to each other. It's like when you look at a picture and try to figure out what's happening in each part of the image.
2. **Feed-Forward Network**: This part is like a calculator that processes the information in a straightforward way. It takes the information from the self-attention part and does some calculations with it.

Now, imagine you're building a tower of these smaller boxes. You start with six of them, which we call "N = 6". Each box has two parts, so you end up with twelve parts in total.

To make sure everything fits together nicely, there are special rules:
- **Residual Connections**: Some

Wir haben in diesem Durchgang auf das Sampling verzichtet und stattdessen eine deterministische Greedy-Search mit einer minimalen repetition_penalty genutzt, um eine stabilere Erklärung zu erhalten.

Nicht gut an der Zusammenfassung ist, dass die technischen Erklärungen erneut unvollständig bleiben, da das Modell zu viel Platz für die Einleitung verbraucht und somit das Token-Limit erreicht, bevor die eigentliche Architektur-Definition abgeschlossen ist.

In [ ]:
output = qwen_generator(
    messages,
    max_new_tokens=300,
    do_sample=True,
    temperature=0.3,
    top_k=0,
    repetition_penalty=1.1
)

print(output[0]['generated_text'][-1]['content'])

Sure! Let's break down this concept into simpler terms:

Imagine you have a big puzzle (our neural network). This puzzle is made up of smaller pieces called "layers". These layers work together to solve problems.

Now, let's focus on one part of our puzzle - the "encoder". It's like a special section that helps us understand information better.

In this section:
1. We start with six similar parts (we call them layers), which we'll number from 0 to 5.
2. Each of these parts has two smaller sections inside it.
3. The first small section uses something called "self-attention". Think of this as a way to pay attention to different parts of the puzzle at once.
4. The second small section is just like a helper that gives quick answers based on what it hears.
5. Both of these small sections use something called "residual connections", which means they add their results back to the original input before moving forward.
6. After both small sections, there's another step called "layer normalizati

Dieses Setting ist zwar wieder relativ ähnlich wie manche vorherigen Einstellungen liefert aber eine witzige und relativ gute Analogie in einfachen Worten, die den Mechanismus beschreiben. Prinzipiell ist vor allem hier beim Qwen Decoder es eher ein trial-and-error bis man gute Parameter gefunden hat und die Outputs auch sehr ähnlich sind und die Qualität eher davon abhängt welche Analogie das Modell gerade verwendet. Kurz gesagt, sollte man vor allem bei deisen älteren Modellen immer die Fakten überprüfen, da viele Informationen erfunden werden

# Conclusion (TODO: Ergänzen nahc merge von anderen Teilen)

### Compare the generated text with the original research paper

Der Originaltext ist extrem dicht, akademisch und voller technischer Fachbegriffe. Die generierten Texte unterscheiden sich je nach Modell drastisch davon:

* Encoder-Decoder Modelle: BART hat den Text beispielsweise kaum neu formuliert, sondern fast wörtlich Sätze aus dem Original kopiert und dabei wichtige Teile wie das Masking oder die Attention oft einfach weggelassen. T5 hat ebenfalls keine guten Zusammenfassungen geliefert.

+ Encoder: RoBERTa hat eigentlich relativ gut funktioniert für das was es können soll (Hat nur Textpassagen kopiert als Antwort).

* Decoder: Qwen hat hier generell gut abgeschnitten und eine relativ gute Zusammenfassung geliefert bei Text-Summarization. Ebenfalls hat Qwen den Prompt, den Abschnitt einfach zu erklären auch mit guten Anekdoten versehen und obwohl es manchmal halluziniert hat, sind nach ein bisschen Tuning der Paramter manch sinnvolle Texte rausgekommen. GPT-2 andererseits hat fast ausschließlich nur halluziniert.

### Discuss how accurate, useful, and coherent the AI-generated text is

Die Nützlichkeit und Kohärenz hing stark vom gewählten Modell und Ziel ab:

* RoBERTa: Sehr akkurat und nützlich beim Finden harter Fakten, aber nutzlos für offene Erklärungsfragen ("What is an encoder?").

* BART & T5: Sowohl BART als auch T5 lieferte zwar kohärente, aber wenig nützliche Zusammenfassungen, da sie nur den Anfang des Textes wiedergaben.

* Qwen2.5-1.5B: Sehr kohärent und flüssig zu lesen, auch wenn die Erklärungen manchmal sehr kreativ wurden. Bei höherer Kreativitiät sind auch oft manche Erklärungen falsch gewesen nur damit sie in die Analogie passten.

### Analyze how different parameters impact text quality, creativity, and relevance.

* Temperature, Top-K & Top-P: Bei niedrigen Werten blieben die Modelle strikt bei den Fakten und produzierten fast schon repetitive Listen. Wurde das Sampling mit höheren Werten aktiviert, stieg die Kreativität stark an, aber bei Modellen wie GPT-2 führte dies zum sofortigen Halluzinieren.

* Repetition Penalty & N-Gram Size: Ohne diese Parameter gerieten T5 und GPT-2 schnell in Endlosschleifen. Setzt man sie zu hoch, zwingt man die KI, krampfhaft Synonyme zu suchen, was in sachlichen Texten oft zu falschen Fachbegriffen führt.

* Max/Min New Tokens: Der Versuch, kleine Modelle durch min_length zu langen Texten zu zwingen, verursachte oft sinnlose Texte mit Wiederholungen. Gleichzeitig führte ein zu niedriges max_new_tokens bei Qwen dazu, dass die Erklärungen mitten im Satz endeten.

### Reflect on limitations - does the model introduce incorrect information? Does it stay on-topic?

* Halluzinationen: Die Modelle haben definitiv Fehlinformationen eingeführt. GPT-2 hat hier am höchsten Halluziniert. Qwen blieb zwar beim Transformer-Thema, verzerrte aber durch die erzwungene Vereinfachung die tatsächliche Funktionsweise technischer Konzepte extrem.

* Fokus-Verlust: Vor allem kleinere oder ältere generative Modelle haben Schwierigkeiten, Prioritäten zu setzen. Sie verbrauchen ihr gesamtes Token-Limit für langatmige Einleitungen und vergessen am Ende, essenzielle Kernpunkte (wie den Attention-Mechanismus) überhaupt zu erwähnen.